In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from toponymy.annotation import NodeId, AnnotationTree, Annotation, AnnotationStore, Executor, IndexedAnnotation

14:44:32 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
14:44:33 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


In [3]:
import numpy as np

In [4]:
from toponymy.tools.notebook_data_load import load_small_newsgroups
newsgroups_df = load_small_newsgroups()
embeddings = np.stack(newsgroups_df["embedding"].values)
document_map = np.stack(newsgroups_df["map"].values)

In [5]:
from toponymy import ToponymyClusterer
clusterer = ToponymyClusterer(min_clusters=4, verbose=True)
clusterer.fit(document_map, embeddings);

Layer 0 found 8 clusters
Layer 1 found 4 clusters


In [6]:
clusterer.cluster_tree_

{(1, 0): [(0, 0)],
 (1, 1): [(0, 1)],
 (1, 2): [(0, 7), (0, 6), (0, 5)],
 (1, 3): [(0, 4)],
 (2, 0): [(1, 0), (1, 1), (1, 2), (1, 3), (0, 2), (0, 3)]}

## Nodes

In [7]:
node = NodeId(1, 0)

In [8]:
node.layer

1

In [9]:
node.cluster

0

In [10]:
node == (1, 0)

True

## AnnotationTree

Here are the assumptions about the condensed tree that must hold.

* There is exactly one root node.
* Every non-root node has exactly one parent.
* Edges may span more than one layer.
* The root is a catch-all node, and will not be considered a cluster node or a part of a layer.
* Layer ids are 0-indexed and there are no empty layers (this assumption is used by the executor)



Can make it from a condensed tree dict

In [11]:
tree = AnnotationTree(clusterer.cluster_tree_)

Or for an object with a `.cluster_tree_`

In [12]:
tree_from_clusterer = AnnotationTree.from_clusterer(clusterer)

Can check equality

In [13]:
tree == tree_from_clusterer

True

Can check if a node is in the tree

In [14]:
(0, 1) in tree

True

The root is not considered to be in the annotation tree

In [15]:
(2, 0) in tree

False

In [16]:
len(tree)

12

### Nodes
Has an iterable of all of the cluster nodes (the non-root nodes)

In [17]:
tree.nodes

(NodeId(0, 0),
 NodeId(0, 1),
 NodeId(0, 2),
 NodeId(0, 3),
 NodeId(0, 4),
 NodeId(0, 5),
 NodeId(0, 6),
 NodeId(0, 7),
 NodeId(1, 0),
 NodeId(1, 1),
 NodeId(1, 2),
 NodeId(1, 3))

In [18]:
len(tree) == len(tree.nodes)

True

### Layers
Gives access to the layers via their ids

In [19]:
tree.n_layers

2

In [20]:
tree.layer_ids

(0, 1)

In [21]:
tree.layer(0)

(NodeId(0, 0),
 NodeId(0, 1),
 NodeId(0, 2),
 NodeId(0, 3),
 NodeId(0, 4),
 NodeId(0, 5),
 NodeId(0, 6),
 NodeId(0, 7))

### Relatives

In [22]:
root_node = (2, 0)
leaf_node = (0, 0)
top_layer_node = (1, 2)

In [23]:
tree.parent(leaf_node)

NodeId(1, 0)

In [24]:
tree.parent(top_layer_node)

In [25]:
tree.children(leaf_node)

In [26]:
tree.children(top_layer_node)

[NodeId(0, 5), NodeId(0, 6), NodeId(0, 7)]

In [27]:
tree.descendants(top_layer_node)

[NodeId(0, 5), NodeId(0, 6), NodeId(0, 7)]

In [28]:
tree.descendants(leaf_node)

[]

The root is hidden from parent/child relationships since it doesn't represent a cluster

In [29]:
tree.parent(root_node)

In [30]:
tree.children(root_node)

In [31]:
tree.descendants(root_node)

[]

If you need to see the root or root children for some reason, there's internal access to them

In [32]:
tree._root

NodeId(2, 0)

In [33]:
tree._root_children

[NodeId(0, 2),
 NodeId(0, 3),
 NodeId(1, 0),
 NodeId(1, 1),
 NodeId(1, 2),
 NodeId(1, 3)]

## Annotation
An `Annotation` is a data store for annotations of the nodes of an `AnnotationTree`

It holds up to one object for each node in an `AnnotationTree`

In [34]:
example = Annotation("example", tree)

Here's an empty Annotation

In [35]:
example.states

mappingproxy({NodeId(0, 0): <AnnotationState.EMPTY: 'empty'>,
              NodeId(0, 1): <AnnotationState.EMPTY: 'empty'>,
              NodeId(0, 2): <AnnotationState.EMPTY: 'empty'>,
              NodeId(0, 3): <AnnotationState.EMPTY: 'empty'>,
              NodeId(0, 4): <AnnotationState.EMPTY: 'empty'>,
              NodeId(0, 5): <AnnotationState.EMPTY: 'empty'>,
              NodeId(0, 6): <AnnotationState.EMPTY: 'empty'>,
              NodeId(0, 7): <AnnotationState.EMPTY: 'empty'>,
              NodeId(1, 0): <AnnotationState.EMPTY: 'empty'>,
              NodeId(1, 1): <AnnotationState.EMPTY: 'empty'>,
              NodeId(1, 2): <AnnotationState.EMPTY: 'empty'>,
              NodeId(1, 3): <AnnotationState.EMPTY: 'empty'>})

or the friendlier version:

In [36]:
example.get(leaf_node)

You can fill in an annotation from a layered list (aka. list of lists indexed by layer id and then cluster id)

In [37]:
# note, no need to cooerce the array to lists, except to be able to show equality below
centroid_vectors_layered_list = [list(clusterer.cluster_layers_[0].centroid_vectors), list(clusterer.cluster_layers_[1].centroid_vectors)]

In [38]:
centroid_annotation = Annotation.from_layered_list("centroids", tree, centroid_vectors_layered_list)

In [39]:
centroid_annotation.states

mappingproxy({NodeId(0, 0): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 3): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 6): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>})

In [40]:
node = NodeId(1, 0)

In [41]:
all(clusterer.cluster_layers_[node.layer].centroid_vectors[node.cluster] == centroid_annotation[node])

True

And get a layered list of lists back from the annotation as long as all of the nodes have been computed. This gives backwards compatibility with the old representations. 

In [42]:
new_layered_list = centroid_annotation.to_layered_list()

In [43]:
centroid_vectors_layered_list == new_layered_list

True

## AnnotationStore

In [44]:
store = AnnotationStore(tree, [centroid_annotation])

In [45]:
store

AnnotationStore(centroids[12/12])

In [46]:
all(store['centroids'][node] == store.centroids[node])

True

In [47]:
all(store.centroids[node] == store.node(node)['centroids'])

True

In [48]:
store.node_states(node)

{'centroids': <AnnotationState.COMPUTED: 'computed'>}

## Annotator

In [49]:
class FirstEntryAnnotator:
    inputs = ("centroids",)
    outputs = ("first_item",)
    algorithm_type = "node-node"

    def annotate(
        self,
        node,
        *,
        centroids # has to match inputs name
    ):
        return {"first_item" : centroids[0]} # keys have to match outputs

## Executor

In [50]:
executor = Executor(store)

In [51]:
failures = executor.run(FirstEntryAnnotator())

In [52]:
failures

{}

In [53]:
store

AnnotationStore(centroids[12/12], first_item[12/12])

In [54]:
store['first_item'][node]

np.float64(-0.010361493309028446)

In [55]:
store['centroids'][node][0]

np.float64(-0.010361493309028446)

## Exemplar

Already have the centroid annotation so now we need: 

* cluster objects annotation
* cluster object vectors annotation

In [56]:
from toponymy.exemplar_texts import diverse_exemplars_by_cluster

In [57]:
cluster_label_vectors_list = [clusterer.cluster_layers_[0].cluster_labels, clusterer.cluster_layers_[1].cluster_labels]

## Create a `cluster_object_indices` annotation

In [58]:
def members_by_cluster(labels, cluster_ids):
    """Positions of the objects in each of cluster_ids, in increasing order."""
    order = np.argsort(labels, kind="stable")
    sorted_labels = labels[order]
    ids = np.asarray(cluster_ids)
    starts = np.searchsorted(sorted_labels, ids, side="left")
    stops = np.searchsorted(sorted_labels, ids, side="right")
    return {int(c): order[a:b] for c, a, b in zip(ids, starts, stops)}

In [59]:
layers = clusterer.cluster_layers_
labels_by_layer = [np.asarray(layer.cluster_labels) for layer in layers]
members_by_layer = [
            members_by_cluster(labels, [node.cluster for node in tree.layer(layer_id)])
            for layer_id, labels in enumerate(labels_by_layer)
        ]
cluster_object_indices = Annotation("cluster_object_indices", tree)
for node in tree.nodes:
    cluster_object_indices[node] = members_by_layer[node.layer][node.cluster]

In [60]:
cluster_object_indices

Annotation(name='cluster_object_indices', 12/12 computed)

In [61]:
cluster_object_indices[(0,0)]

array([  2,   3,  19,  47,  49,  54,  58,  66,  74,  82,  88,  96,  98,
       101, 111, 115, 117, 133, 134, 149])

In [62]:
store.add(cluster_object_indices)

### Create `cluster_object` annotation

There is a special kind of array-backed annotation that lets you create an annotation by giving an array and a sequence of indices into the array per node so that at each node the `annotation[node] = array[node_indices]`, it's called an `IndexedAnnotation`. Let's try that out here. Note: An `IndexedAnnotation` can't be an output of an `Annotator`. 

In [63]:
object_list = newsgroups_df["post"].str.strip().values

In [64]:
cluster_object_annotation = IndexedAnnotation("cluster_objects", cluster_object_indices, object_list)

In [65]:
cluster_object_annotation

IndexedAnnotation(name='cluster_objects', 12/12 computed)

In [66]:
cluster_object_annotation.states

mappingproxy({NodeId(0, 0): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 3): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 6): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>})

In [67]:
[f"{object[:30]}... " for object in cluster_object_annotation[0, 0]]

['I have tickets for the TB Gian... ',
 'This just in...\n\nNolan Ryan hu... ',
 'Disclaimer -- This is for fun.... ',
 '-=> Quoting The Logistician to... ',
 'NL:\n\nChicago Wait Til Next Yea... ',
 'Is there any judgement call on... ',
 "Sorry, I still don't buy it.  ... ",
 'OK, you guys stirred up my chi... ',
 'Well, it looks like, just as D... ',
 '[about the infield fly rule]\n\t... ',
 'Not totally true.  For the pas... ',
 'Gretzky, Lemieux, Gilmour etc ... ',
 'Rob Niedermayer is a forward (... ',
 '#21\tPETER AHOLA\t\tSeason: 2nd\nA... ',
 'that\nhis\npositive.\n\n\tClint Mal... ',
 "Not if you've scored four runs... ",
 'ya know that kind of funny cau... ',
 '...\n...\n\n\n\n\n\nWe must have been... ',
 'Tampa Bay                     ... ',
 "I agree. I'll delete more as w... "]

In [68]:
store.add(cluster_object_annotation)

### Create `cluster_object_vectors_annotation` from `object_vectors`

In [69]:
object_vectors = embeddings

In [70]:
cluster_object_vectors_annotation = IndexedAnnotation("cluster_object_vectors", cluster_object_indices, object_vectors)

In [71]:
cluster_object_vectors_annotation.states

mappingproxy({NodeId(0, 0): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 3): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 6): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
              NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>})

In [72]:
cluster_object_vectors_annotation[0,0]

array([[-0.03250182, -0.00795813,  0.00175691, ...,  0.0059225 ,
         0.00409398, -0.01085705],
       [-0.02435001,  0.0376508 , -0.00252953, ..., -0.02046058,
        -0.0516339 , -0.02025957],
       [-0.03926771,  0.04878276,  0.00287326, ...,  0.01039253,
         0.04837336, -0.01026004],
       ...,
       [-0.00765608, -0.03975155, -0.02702211, ...,  0.03710938,
         0.06257425,  0.01677064],
       [-0.03680376, -0.03056411, -0.02371935, ..., -0.02412126,
         0.00309259, -0.01114765],
       [-0.01758591,  0.09526303,  0.01307244, ...,  0.08918627,
         0.02194043, -0.01463626]], shape=(20, 768))

In [73]:
store.add(cluster_object_vectors_annotation, replace=True)

## Compute exemplars

raw version

In [74]:
store

AnnotationStore(centroids[12/12], first_item[12/12], cluster_object_indices[12/12], cluster_objects[12/12], cluster_object_vectors[12/12])

In [75]:
posts = store["cluster_objects"][node]
print(f"{len(posts)} posts in node:{node}")
for i, post in enumerate(posts[:5]):
    print(f"{i}: {' '.join(post.split())[:80]}...")

20 posts in node:(1, 3)
0: A list of options that would be useful. They can be existing options on a car, o...
1: Since we're on the subject of brakes.... does anyone know why a 4WD Vauxhall/ Op...
2: Ron> Viper also sells some fancy field disturbance sensor that Ron> supposedly d...
3: Highway 12 is a great road. Be sure to stop by the Anasazi (sp) village museum n...
4: They detect the oscillator operating in the detector. Saw a story about their us...


In [76]:
null_topic = np.mean(object_vectors, axis=0)

In [77]:
results = []
indices = []

for node in tree.nodes:
    layer = clusterer.cluster_layers_[node.layer]
    node_result = diverse_exemplars_by_cluster(cluster_objects=np.array(store["cluster_objects"][node]),
                                               cluster_centroid=store["centroids"][node],
                                               cluster_object_vectors=store["cluster_object_vectors"][node], 
                                               null_topic=null_topic,
                                               n_exemplars=layer.n_exemplars,
                                               diversify_alpha=layer.exemplars_diversify_alpha,
                                               object_to_text_function=layer.object_to_text_function,
                                               verbose=layer.verbose,
                                               show_progress_bar=layer.show_progress_bar)
    chosen_exemplars, exemplar_order, chosen_indices  = node_result
    cluster_mask = layer.cluster_labels == node.cluster
    original_indices = np.where(cluster_mask)[0]
    chosen_original_indices = [
        original_indices[exemplar_order[i]] for i in chosen_indices
    ]
    
    results.append(chosen_exemplars)
    indices.append(chosen_original_indices)


In [78]:
layer.exemplars_diversify_alpha

1.0

In [79]:
from toponymy.exemplar_texts import DiverseExemplarAnnotator

In [80]:
failures = executor.run(DiverseExemplarAnnotator(null_topic=null_topic,
                                                 n_exemplars=8,
                                                 diversify_alpha=layer.exemplars_diversify_alpha,
                                                 object_to_text_function=layer.object_to_text_function),
                        input_names={"cluster_centroid": "centroids"})
failures

{}

In [81]:
node_values = store.node(NodeId(0, 0))
for idx, text in zip(node_values["exemplar_original_indices"], node_values["exemplars"]):
    print(f"{idx}: {' '.join(text.split())[:80]}...")

74: Well, it looks like, just as Doug trumped Tim, beating him to the net with his d...
66: OK, you guys stirred up my childhood memories, so I went and did some research o...
101: #21 PETER AHOLA Season: 2nd Acquired: '92-93, trade with Pittsburgh for future c...
54: Is there any judgement call on the part of the scorer for sac fly RBI's? This is...
134: Tampa Bay 1 1 0--2 Philadelphia 3 2 1--6 First period 1, Philadelphia, Carkner 3...
111: that his positive. Clint Malarchuk What has happened to him since? Assigned to S...
115: Not if you've scored four runs, you don't! Why strain even the best pitching sta...
3: This just in... Nolan Ryan hurt his right knee in the 4th inning of the Rangers-...


And these results are the same as those generated by our previous results calculation.

In [82]:
for idx, text in zip(indices[0], results[0]):
    print(f"{idx}: {' '.join(text.split())[:80]}...")

74: Well, it looks like, just as Doug trumped Tim, beating him to the net with his d...
66: OK, you guys stirred up my childhood memories, so I went and did some research o...
101: #21 PETER AHOLA Season: 2nd Acquired: '92-93, trade with Pittsburgh for future c...
54: Is there any judgement call on the part of the scorer for sac fly RBI's? This is...
134: Tampa Bay 1 1 0--2 Philadelphia 3 2 1--6 First period 1, Philadelphia, Carkner 3...
111: that his positive. Clint Malarchuk What has happened to him since? Assigned to S...
115: Not if you've scored four runs, you don't! Why strain even the best pitching sta...
3: This just in... Nolan Ryan hurt his right knee in the 4th inning of the Rangers-...


# Can use `.from_clusterer` to get the initial `AnnotationStore`

In [83]:
new_store = AnnotationStore.from_clusterer(clusterer, object_list, object_vectors)

In [84]:
new_store

AnnotationStore(cluster_object_indices[12/12], cluster_objects[12/12], cluster_centroid[12/12], cluster_object_vectors[12/12])

In [85]:
new_store["cluster_centroid"][(0,0)]

array([-1.03614933e-02,  1.44754660e-02,  7.03579400e-03,  4.55402070e-03,
       -1.78067970e-02, -1.54497339e-02, -3.03466713e-02,  4.59112967e-03,
       -3.94506354e-02, -2.53642215e-02,  2.22228956e-02,  2.30358896e-02,
        1.84383085e-02, -1.51750268e-02,  1.32262419e-02, -6.39660904e-03,
       -2.60208459e-02, -1.27288801e-02,  2.35251141e-02,  1.20141171e-04,
       -2.49168110e-03,  8.00823662e-03, -1.77270293e-02,  1.23710510e-02,
       -5.45138177e-03,  2.45723501e-04,  2.95007439e-02, -1.22677118e-02,
       -1.52631403e-02, -2.76844425e-02, -9.93311192e-03,  6.77673371e-03,
       -1.04914314e-02, -2.35398327e-02,  2.00108988e-06, -1.08542800e-02,
        2.47618027e-02, -5.33511581e-03, -8.67506663e-03,  6.99183642e-03,
        6.28974067e-03, -3.50769975e-03,  2.45528850e-02,  2.02116382e-02,
       -1.52671702e-03, -8.22793925e-05,  2.97518244e-03, -9.29565962e-03,
       -6.38745462e-03,  8.26569257e-03,  1.62473315e-02,  1.34872744e-02,
       -1.44480288e-02, -

In [86]:
all(store["centroids"][(0,0)] == new_store["cluster_centroid"][(0,0)])

True

In [87]:
(store["cluster_object_vectors"][(0,0)] == new_store["cluster_object_vectors"][(0,0)]).all()

np.True_